In [1]:
from utils import createSparkSession, getThroughputRate, read_csv_from_s3, write_postgres_table
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ByteType, ShortType

In [2]:
spark, stagemetrics = createSparkSession('P1M1')

26/11/07 09:51:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## M0-2 Stage as text

In [3]:
stagemetrics.begin()

In [4]:
aisles_schema = StructType([
    StructField('aisle_id', StringType(), False),
    StructField('aisle', StringType(), True),
    StructField('bad_record', StringType(), True),
])

departments_schema = StructType([
    StructField('department_id', StringType(), False),
    StructField('department', StringType(), True),
    StructField('bad_record', StringType(), True),
])

order_products_schema = StructType([
    StructField('order_id', StringType(), False),
    StructField('product_id', StringType(), False),
    StructField('add_to_cart_order', StringType(), True),
    StructField('reordered', StringType(), True),
    StructField('bad_record', StringType(), True),
])

products_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('product_name', StringType(), True),
    StructField('aisle_id', StringType(), True),
    StructField('department_id', StringType(), True),
    StructField('bad_record', StringType(), True),
])

orders_schema = StructType([
    StructField('order_id', StringType(), False),
    StructField('user_id', StringType(), False),
    StructField('eval_set', StringType(), True),
    StructField('order_number', StringType(), True),
    StructField('order_dow', StringType(), True),
    StructField('order_hour_of_day', StringType(), True),
    StructField('days_since_prior_order', StringType(), True),
    StructField('bad_record', StringType(), True),
])

In [5]:
options = {
    'columnNameOfCorruptRecord': 'bad_record',
    'escape':'"'
}

In [6]:
aisles_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/aisles.csv", schema=aisles_schema, options=options)
departments_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/departments.csv", schema=departments_schema, options=options)
order_products__prior_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__prior.csv", schema=order_products_schema, options=options)
order_products__train_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__train.csv", schema=order_products_schema, options=options)
orders_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/orders.csv", schema=orders_schema, options=options)
products_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/products.csv", schema=products_schema, options=options)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [7]:
products_df.filter('bad_record is not null').show()

[Stage 0:>                                                          (0 + 1) / 1]

+----------+------------+--------+-------------+----------+
|product_id|product_name|aisle_id|department_id|bad_record|
+----------+------------+--------+-------------+----------+
+----------+------------+--------+-------------+----------+



In [8]:
products_df.filter('product_id = 6816').show()

+----------+--------------------+--------+-------------+----------+
|product_id|        product_name|aisle_id|department_id|bad_record|
+----------+--------------------+--------+-------------+----------+
|      6816|Scotch Kids 5\" S...|      87|           17|      NULL|
+----------+--------------------+--------+-------------+----------+



In [9]:
order_products_df = order_products__prior_df.union(order_products__train_df)

In [9]:
# write_postgres_table(aisles_df, 'staging_text.aisles')
# write_postgres_table(departments_df, 'staging_text.departments')
# write_postgres_table(order_products_df, 'staging_text.order_products')
# write_postgres_table(orders_df, 'staging_text.orders')
# write_postgres_table(products_df, 'staging_text.products')

In [10]:
stagemetrics.end()

In [11]:
stagemetrics.print_report()
stagemetrics.print_memory_report()
getThroughputRate(stagemetrics.aggregate_stagemetrics())


Scheduling mode = FIFO
Spark Context default degree of parallelism = 9

Aggregated Spark stage metrics:
numStages => 7
numTasks => 29
elapsedTime => 130722 (2.2 min)
stageDuration => 129807 (2.2 min)
executorRunTime => 1074982 (18 min)
executorCpuTime => 129266 (2.2 min)
executorDeserializeTime => 4311 (4 s)
executorDeserializeCpuTime => 2255 (2 s)
resultSerializationTime => 35 (35 ms)
jvmGCTime => 41846 (42 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
resultSize => 26812 (26.2 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 37389408
bytesRead => 719079486 (685.8 MB)
recordsWritten => 37290032
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 0
shuffleTotalBlocksFetched => 0
shuffleLocalBlocksFetched => 0
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 0 (0 Bytes)
shuffleLocalBytesRead => 0 (0 Bytes)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWr

26/11/05 05:18:28 WARN StageMetrics: No executor metrics found for stage 6 and metric JVMHeapMemory.
26/11/05 05:18:28 WARN StageMetrics: No executor metrics found for stage 6 and metric OnHeapExecutionMemory.


## M0-3 Parsing fixes

In [3]:
stagemetrics.begin()

In [3]:
aisles_schema = StructType([
    StructField('aisle_id', IntegerType(), False),
    StructField('aisle', StringType(), True),
    StructField('bad_record', StringType(), True),
])

departments_schema = StructType([
    StructField('department_id', IntegerType(), False),
    StructField('department', StringType(), True),
    StructField('bad_record', StringType(), True),
])

order_products_schema = StructType([
    StructField('order_id', IntegerType(), False),
    StructField('product_id', IntegerType(), False),
    StructField('add_to_cart_order', IntegerType(), True),
    StructField('reordered', ByteType(), True),
    StructField('bad_record', StringType(), True),
])

products_schema = StructType([
    StructField('product_id', IntegerType(), False),
    StructField('product_name', StringType(), True),
    StructField('aisle_id', IntegerType(), True),
    StructField('department_id', IntegerType(), True),
    StructField('bad_record', StringType(), True),
])

orders_schema = StructType([
    StructField('order_id', IntegerType(), False),
    StructField('user_id', IntegerType(), False),
    StructField('eval_set', StringType(), True),
    StructField('order_number', IntegerType(), True),
    StructField('order_dow', IntegerType(), True),
    StructField('order_hour_of_day', IntegerType(), True),
    StructField('days_since_prior_order', DoubleType(), True),
    StructField('bad_record', StringType(), True),
])

In [4]:
options = {
    'columnNameOfCorruptRecord': 'bad_record',
    'escape':'"'
}

In [5]:
aisles_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/aisles.csv", schema=aisles_schema, options=options)
departments_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/departments.csv", schema=departments_schema, options=options)
order_products__prior_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__prior.csv", schema=order_products_schema, options=options)
order_products__train_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__train.csv", schema=order_products_schema, options=options)
orders_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/orders.csv", schema=orders_schema, options=options)
products_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/products.csv", schema=products_schema, options=options)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


products csv has a bad record; issue is csv is standard csv with double quote as escape character

In [7]:
products_df.filter('bad_record is not NULL').show()

[Stage 0:>                                                          (0 + 1) / 1]

+----------+------------+--------+-------------+----------+
|product_id|product_name|aisle_id|department_id|bad_record|
+----------+------------+--------+-------------+----------+
+----------+------------+--------+-------------+----------+



In [8]:
products_df.filter('product_id = 6816').show()

+----------+--------------------+--------+-------------+----------+
|product_id|        product_name|aisle_id|department_id|bad_record|
+----------+--------------------+--------+-------------+----------+
|      6816|Scotch Kids 5\" S...|      87|           17|      NULL|
+----------+--------------------+--------+-------------+----------+



In [6]:
order_products_df = order_products__prior_df.union(order_products__train_df)

In [12]:
# write_postgres_table(aisles_df, 'staging.aisles')
# write_postgres_table(departments_df, 'staging.departments')
# write_postgres_table(order_products_df, 'staging.order_products')
# write_postgres_table(orders_df, 'staging.orders')
# write_postgres_table(products_df, 'staging.products')

In [13]:
stagemetrics.end()

In [14]:
stagemetrics.print_report()
stagemetrics.print_memory_report()
getThroughputRate(stagemetrics.aggregate_stagemetrics())


Scheduling mode = FIFO
Spark Context default degree of parallelism = 8

Aggregated Spark stage metrics:
numStages => 2
numTasks => 2
elapsedTime => 4198 (4 s)
stageDuration => 4064 (4 s)
executorRunTime => 3062 (3 s)
executorCpuTime => 2157 (2 s)
executorDeserializeTime => 806 (0.8 s)
executorDeserializeCpuTime => 683 (0.7 s)
resultSerializationTime => 7 (7 ms)
jvmGCTime => 89 (89 ms)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
resultSize => 1814 (1814 Bytes)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 49689
bytesRead => 4333906 (4.1 MB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 0
shuffleTotalBlocksFetched => 0
shuffleLocalBlocksFetched => 0
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 0 (0 Bytes)
shuffleLocalBytesRead => 0 (0 Bytes)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWritten => 0 (0 Bytes)
shuffleRecordsWrit

26/11/07 07:01:49 WARN StageMetrics: No executor metrics found for stage 0 and metric JVMHeapMemory.
26/11/07 07:01:49 WARN StageMetrics: No executor metrics found for stage 0 and metric OnHeapExecutionMemory.
26/11/07 07:01:49 WARN StageMetrics: No executor metrics found for stage 1 and metric JVMHeapMemory.
26/11/07 07:01:49 WARN StageMetrics: No executor metrics found for stage 1 and metric OnHeapExecutionMemory.


## M0-5 Profiling

In [8]:
from data_profiling import ProfileReport

In [9]:
ProfileReport(aisles_df).to_file('/home/ubuntu/work_dir/profiling/aisles.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:14:38 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:14:41 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
ProfileReport(departments_df).to_file('/home/ubuntu/work_dir/profiling/departments.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:14:53 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:14:54 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
ProfileReport(order_products__prior_df).to_file('/home/ubuntu/work_dir/profiling/order_products__prior.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:16:23 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:16:45 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:17:04 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:17:17 WARN CacheManager: Asked to cache already cached data.        
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
ProfileReport(order_products__train_df).to_file('/home/ubuntu/work_dir/profiling/order_products__train.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:18:37 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:18:41 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:18:45 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:18:47 WARN CacheManager: Asked to cache already cached data.
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
ProfileReport(orders_df).to_file('/home/ubuntu/work_dir/profiling/orders.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:19:05 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:15 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:20 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:23 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:27 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:30 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:34 WARN CacheManager: Asked to cache already cached data.        
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
ProfileReport(products_df).to_file('/home/ubuntu/work_dir/profiling/products.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:19:49 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:50 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:52 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:54 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
spark.stop()